# Imports

In [ ]:
import keras
import numpy as np

# Model

In [ ]:
def get_name(prefix: str | None = None, suffix: str | None = None, separator: str = '_') -> str | None:
    return prefix and prefix + separator + suffix or suffix or None

In [ ]:
def get_conv_bn(
    x: keras.KerasTensor,
    channels: int,
    kernel_size: int,
    padding: str,
    strides: int,
    activation: str | keras.layers.Activation | None = 'silu',
    use_bias: bool = True,
    name: str | None = None
) -> keras.KerasTensor:
    """Applies 2D Convolution followed by BatchNormalization and (possibly) `activation`

    Arguments:
        x: tensor to apply the operation to
        channels: number of output channels
        kernel_size: kernel size of the convolution
        padding: either 'same' or 'valid', padding mode for the convolution
        strides: stride size for the convolution
        activation: activation to apply after batch normalization
        use_bias: whether to learn bias in the convolution
        name: name of the operation

    Returns:
        The resulting tensor"""

    x = keras.layers.Conv2D(channels, kernel_size=kernel_size, padding=padding, use_bias=use_bias, strides=strides, name=get_name(name, 'conv'))(x)
    x = keras.layers.BatchNormalization(name=get_name(name, 'bn'))(x)
    
    if activation is not None:
        x = keras.layers.Activation(activation, name=get_name(name, 'activation'))(x)

    return x

In [ ]:
def get_bottleneck(
    x: keras.KerasTensor,
    has_skip_connection: bool,
    activation: str | keras.layers.Activation | None = 'silu',
    name: str | None = None
) -> keras.KerasTensor:
    """Applies bottleneck block (see YOLOv5 architecture for details)

    Arguments:
        x: tensor to apply the operation to
        has_skip_connection: whether to use skip connection
        activation: activation to apply in internal convolutions
        name: name of the operation

    Returns:
        The resulting tensor"""
    
    channels = x.shape[-1]
    y = get_conv_bn(x, channels, kernel_size=1, padding='valid', strides=1, activation=activation, name=get_name(name, 'conv1'))
    y = get_conv_bn(y, channels, kernel_size=3, padding='same',  strides=1, activation=activation, name=get_name(name, 'conv2'))
    
    if has_skip_connection:
        y = keras.layers.Add(name=get_name(name, 'skip_connection'))([y, x])

    return y

In [ ]:
def get_C3_block(
    x: keras.KerasTensor,
    size: int,
    half_size: bool,
    activation: str | keras.layers.Activation | None = 'silu',
    name: str | None = None
) -> keras.KerasTensor:
    """Applies C3 block (see YOLOv5 architecture for details)

    Arguments:
        x: tensor to apply the operation to
        size: number of bottlenecks
        half_size: whether the output channels are half of the input ones
        activation: activation to apply in internal convolutions
        name: name of the operation

    Returns:
        The resulting tensor"""

    channels = x.shape[-1] // 2
    y = get_conv_bn(x, channels, kernel_size=1, padding='valid', strides=1, activation=activation, name=get_name(name, 'conv1'))
    x = get_conv_bn(x, channels, kernel_size=1, padding='valid', strides=1, activation=activation, name=get_name(name, 'conv2'))
    
    for i in range(size):
        x = get_bottleneck(x, half_size, activation=activation, name=get_name(name, f'bottleneck_{i + 1}'))

    x = keras.layers.Concatenate(name=get_name(name, 'concatenate'))([x, y])

    multiplier = 1
    if not half_size:
        multiplier = 2

    x = get_conv_bn(x, channels * multiplier, kernel_size=1, padding='valid', strides=1, activation=activation, name=get_name(name, 'out_conv')) 

    return x     

In [ ]:
def get_yolo_backbone(
    channels: int,
    block_sizes: list[int],
    pyramid_sizes: list[int],
    activation: str | keras.layers.Activation | None = 'silu',
    name: str | None = None
) -> keras.Model:
    """Builds YOLOv5 style model.

    Arguments:
        channels: number of channels for the first convolution
        block_sizes: number of bottleneck layers in each down block
        pyramid_sizes: number of bottleneck layers in each pyramid level
        activation: activation to apply in internal convolutions
        name: name of the model

    Return:
        The model"""

    inputs = keras.layers.Input((None, None, 3), dtype='uint8', name='input')
    x = keras.layers.Rescaling(scale=1. / 127.5, offset=-1, name='rescaling')(inputs)
    
    x = get_conv_bn(x, channels, kernel_size=6, padding='same', strides=2, activation=activation, name='stem')

    levels = []
    for i, s in enumerate(block_sizes):
        channels *= 2
        block_name = f'block_{i + 1}'
        
        x = get_conv_bn(x, channels, kernel_size=3, padding='same', strides=2, activation=activation, name=get_name(block_name, 'stem'))
        x = get_C3_block(x, block_sizes[i], half_size=False, activation=activation, name=get_name(block_name, 'C3'))
        levels.append(x)

    sppf_name = 'sppf'
    x = get_conv_bn(x, channels // 2, kernel_size=1, padding='same', strides=1, activation=activation, name=get_name(sppf_name, 'in_conv'))

    max_pools = [x]
    for i in range(3):
        x = keras.layers.MaxPooling2D(pool_size=5, strides=1, padding='same', name=get_name(sppf_name, f'max_pool_{i + 1}'))(x)
        max_pools.append(x)

    x = keras.layers.Concatenate(name=get_name(sppf_name, 'concatenate'))(max_pools)
    x = get_conv_bn(x, channels, kernel_size=1, padding='valid', strides=1, activation=activation, name=get_name(sppf_name, 'out_conv'))

    pyramid_levels = [x]
    pyramid_name = 'up_pyramid'
    
    for i, j in enumerate(pyramid_sizes):
        block_name = get_name(pyramid_name, f'block_{i + 1}')
        
        y = levels[-i - 2]
        x = pyramid_levels[-1] = get_conv_bn(pyramid_levels[-1], y.shape[-1], kernel_size=1, padding='valid', strides=1, activation=activation, name=get_name(block_name, 'conv'))
        
        x = keras.layers.UpSampling2D(size=2, interpolation='nearest', name=get_name(block_name, 'upsampling'))(x)
        x = keras.layers.Concatenate(name=get_name(block_name, 'concatenate'))([x, y])

        x = get_C3_block(x, j, half_size=True, activation=activation, name=get_name(block_name, 'C3'))

        pyramid_levels.append(x)

    outputs = [pyramid_levels[-1]]
    pyramid_name = 'down_pyramid'

    for i, j in enumerate(pyramid_sizes[::-1]):
        channels = outputs[-1].shape[-1]
        block_name = get_name(pyramid_name, f'block_{i + 1}')
        
        x = get_conv_bn(outputs[-1], channels, kernel_size=3, padding='same', strides=2, activation=activation, name=get_name(block_name, 'conv'))

        x = keras.layers.Concatenate(name=get_name(block_name, 'concatenate'))([x, pyramid_levels[-i - 2]])

        x = get_C3_block(x, j, half_size=False, activation=activation, name=get_name(block_name, 'C3'))

        outputs.append(x)

        
    return keras.Model(inputs=inputs, outputs=outputs, name=name)

# Neck

In [ ]:
def depthwise_separable_conv(
    x: keras.KerasTensor,
    channels: int,
    kernel_size: int,
    strides: int,
    activation: str | keras.layers.Activation | None = 'silu',
    name: str | None = None
) -> keras.KerasTensor:
    """Applies depthwise separable convolution.

    Arguments:
        x: tensor to apply the operation to
        channels: number of output channels
        kernel_size: kernel size for the depthwise convolution
        strides: stride size for the convolutions
        activation: activation to apply in internal convolutions
        name: name of the operation

    Returns:
        The resulting tensor"""

    x = keras.layers.DepthwiseConv2D(kernel_size, use_bias=False, strides=strides, padding='same', name=get_name(name, 'depthwise_conv'))(x)
    x = keras.layers.BatchNormalization(name=get_name(name, 'depthwise_batchnorm'))(x)

    if activation is not None:
        x = keras.layers.Activation(activation=activation, name=get_name(name, 'depthwise_activation'))(x)

    x = get_conv_bn(x, channels, kernel_size=1, use_bias=False, strides=strides, padding='valid', activation=activation, name=get_name(name, 'pointwise'))

    return x

In [ ]:
def depthwise_separable_convs(
    x: keras.KerasTensor,
    channels: int,
    out_channels: int,
    n_convs: int,
    activation: str | keras.layers.Activation | None = 'silu',
    name: str | None = None
):
    """Applies depthwise separable convolutions.

    Arguments:
        x: tensor to apply the operation to
        channels: number of output channels
        out_channels: number of output channels
        n_convs: number of convolutions
        activation: activation to apply in internal convolutions
        name: name of the operation

    Returns:
        The resulting tensor"""
    
    for i in range(n_convs):
        x = depthwise_separable_conv(x, channels, 3, 1, name=get_name(name, f'conv_{i}'), activation=activation)

    x = keras.layers.Conv2D(out_channels, 3, padding='same', name=get_name(name, 'out_conv'))(x)

    return x

In [ ]:
def yolo_head(
    x: list[keras.KerasTensor],
    n_anchors_per_pixel: int,
    n_convs: int,
    n_reg: int,
    n_classes: int,
    activation: str | keras.layers.Activation | None = 'silu',
    name: str | None = None,
) -> list[tuple[keras.KerasTensor, keras.KerasTensor]]:
    """Applies YOLO detection head.

    Arguments:
        x: list of outputs from backbone levels
        n_anchors_per_pixel: number of anchors per output pixel
        n_convs: number of convolutions for each output
        n_reg: number of DFL points
        n_classes: number of detected classes
        activation: activation to apply in internal convolutions
        name: name of the operation

    Returns:
        list of pairs predicted boxes and predicted classes"""

    outputs = []

    for i, y in enumerate(x):
        boxes = depthwise_separable_convs(y, max(16, y.shape[-1] // n_reg, n_reg * 4),  n_anchors_per_pixel * n_reg * 4, n_convs, activation, get_name(name, f'prediction_box_{i}'))
        boxes = keras.layers.Reshape((-1, 4, n_reg), name=get_name(name, f'reshape_{i}'))(boxes)
        
        classes = depthwise_separable_convs(y,  max(y.shape[-1], min(n_classes, 128)), n_classes, n_convs, activation, get_name(name, f'prediction_class_{i}'))
        classes = keras.layers.Reshape((-1, n_classes), name=get_name(name, f'classes_{i}_flatten'))(classes)        

        outputs.append((boxes, classes))

    
    return outputs

# Build model

You can use smaller model to make training easier

In [ ]:
backbone = get_yolo_backbone(64, [3, 6, 9, 3], [3, 3], name='detection_backbone')

In [ ]:
predictions = yolo_head(backbone.outputs, n_anchors_per_pixel=2, n_convs=2, n_reg=8, n_classes=1, name='detection_head', activation='silu') # don't forget to change the number of classes

In [ ]:
model = keras.Model(inputs=backbone.inputs, outputs=predictions, name='detection_model')

In [ ]:
model.summary()

# Dataset

In [ ]:
def iou(a: keras.KerasTensor, b: keras.KerasTensor) -> keras.KerasTensor:
    """Computes IOS between boxes"""

    xt = keras.ops.maximum(a[..., 0], b[..., 0])
    yt = keras.ops.maximum(a[..., 1], b[..., 1])
    xb = keras.ops.minimum(a[..., 2], b[..., 2])
    yb = keras.ops.minimum(a[..., 3], b[..., 3])

    intersection = keras.ops.maximum(xb - xt, 0) * keras.ops.maximum(yb - yt, 0)
    sa = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    sb = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])

    union = sa + sb - intersection

    return intersection / (union + 1e-10)

In [ ]:
def atss(
    boxes: keras.KerasTensor | np.ndarray,
    labels: keras.KerasTensor | np.ndarray,
    anchors: list[keras.KerasTensor] | list[np.ndarray],
    k: int = 9
) -> tuple[list[keras.KerasTensor], list[keras.KerasTensor]]:
    """Adaptive Training Sample Selection
    
    Arguments:
        boxes: 2D array of ground-truth boxes specifications (N, 4) in xyxy format
        labels: 1D array of box labels (N, )
        anchors: list of 2D array anchor specifications (M, 4) in xyxy format  
        k: number of closest anchors in each level for ground-truth boxes     
    Returns:
        labels: list of 1D arrays of labels assigned to each anchor
        offsets: list of 2D arrays of offsests assigned to each anchor
    """
    
    boxes = keras.ops.cast(boxes, 'float32')
    anchors = [keras.ops.cast(anchor, 'float32') for anchor in anchors]
    
    boxes_centers = keras.ops.stack([boxes[:, 2] + boxes[:, 0], boxes[:, 3] + boxes[:, 1]], axis=-1) / 2
    anchors_centers = [
        keras.ops.stack([anchor[:, 2] + anchor[:, 0], anchor[:, 3] + anchor[:, 1]], axis=-1) / 2 
        for anchor in anchors
    ]

    distances_between_centers = [        
        keras.ops.norm(keras.ops.expand_dims(boxes_centers, axis=1) - keras.ops.expand_dims(anchor_centers, axis=0), axis=-1) 
        for anchor_centers in anchors_centers
    ]

    min_indices = [keras.ops.top_k(-i, k=k, sorted=False)[1] for i in distances_between_centers]

    selected_anchors = keras.ops.concatenate([
        keras.ops.reshape(keras.ops.take(anchor, keras.ops.reshape(i, (-1, )), axis=0), (-1, k, 4)) 
        for i, anchor in zip(min_indices, anchors)
    ], axis=1)

    selected_anchor_centers = keras.ops.concatenate([
        keras.ops.tile(keras.ops.reshape(keras.ops.take(centers, keras.ops.reshape(i, (-1, )), axis=0), (-1, k, 2)), (1, 1, 2))
        for i, centers in zip(min_indices, anchors_centers)
    ], axis=1)

    # not too efficient, but should be easier to understand

    expanded_boxes = keras.ops.expand_dims(boxes, axis=1)
    l = selected_anchor_centers[..., 0] - expanded_boxes[..., 0]
    t = selected_anchor_centers[..., 1] - expanded_boxes[..., 1]
    r = expanded_boxes[..., 2] - selected_anchor_centers[..., 2]
    b = expanded_boxes[..., 3] - selected_anchor_centers[..., 3]

    is_in_box = keras.ops.min(keras.ops.stack([l, t, r, b], axis=-1), axis=-1) > 0    
    
    ious = iou(selected_anchors, keras.ops.expand_dims(boxes, axis=1))
    mean = keras.ops.mean(ious, axis=-1)
    std  = keras.ops.std(ious, axis=-1)

    threshold = mean + std

    mask = keras.ops.logical_and(ious >= keras.ops.expand_dims(threshold, axis=-1), is_in_box)

    labels = keras.ops.tile(keras.ops.expand_dims(labels, axis=-1), (1, len(anchors) * k))

    # normalize offsets, so it doesn't depend on the anchor size
    
    offsets = expanded_boxes - selected_anchor_centers
    offsets = offsets / keras.ops.tile(keras.ops.stack([selected_anchors[..., 2] - selected_anchors[..., 0], selected_anchors[..., 3] - selected_anchors[..., 1]], axis=-1), (1, 1, 2))

    # flatten everything for easier assignment
    
    anchors_per_level = [keras.ops.shape(i)[0] for i in anchors]
    total_anchros = keras.ops.sum(anchors_per_level)    

    # adjust indices to index the flattened anchors
    min_indices = keras.ops.stack(min_indices, axis=1) + keras.ops.reshape(keras.ops.cumsum(anchors_per_level) - anchors_per_level, (1, -1, 1))

    mask = keras.ops.reshape(mask, (-1, ))
    labels = keras.ops.reshape(labels, (-1, ))[mask]
    offsets = keras.ops.reshape(offsets, (-1, 4))[mask]
    min_indices = keras.ops.reshape(min_indices, (-1, 1))[mask]

    offsets = keras.ops.split(keras.ops.scatter(min_indices, offsets, (total_anchros, 4)), keras.ops.cumsum(anchors_per_level)[:-1])
    labels = keras.ops.split(keras.ops.scatter(min_indices, labels, (total_anchros,)), keras.ops.cumsum(anchors_per_level)[:-1])

    return offsets, labels

In [ ]:
def get_anchors(height: keras.KerasTensor, width: keras.KerasTensor, size: int, scales: list[float], stride: int) -> keras.KerasTensor:
    """Get anchors for a model output of size `height` x `width`

    Arguments:
        height: model output height
        width: model output width
        size: anchor size
        scales: anchor scales
        stride: anchor stride        

    Returns:
        The anchors"""

    shift_x = keras.ops.arange(width, dtype='int32') * stride
    shift_y = keras.ops.arange(height, dtype='int32') * stride

    answ_result = []
    for i, scale in enumerate(scales):
        anchor = int(-size // 2 * scale)

        coords_x = shift_x + anchor
        coords_y = shift_y + anchor        

        x, y = keras.ops.meshgrid(coords_x, coords_y)

        ss = int(size * scale)
        result = keras.ops.stack([x, y, x + ss, y + ss], axis=-1)

        last_result = keras.ops.reshape(result, (-1, 4))
        answ_result.append(last_result)

    outputs = keras.ops.concatenate(answ_result, axis=1)
    outputs = keras.ops.reshape(outputs, (-1, 4))

    return outputs

In [ ]:
anchors = [get_anchors(*output.shape[1:3], i, [1, 2], j) for output, i, j in zip(backbone(np.zeros((1, 640, 640, 3))), [16, 64, 256], [8, 16, 32])] # adjust if need be

Build you dataset. Images should be cropped and/or resized to a common size (don't forget to adjust boxes too, can some augmentations be applied?). Keep in mind that ground truth values are corner offsets relative to anchor center w.r.t anchor size and classes for each of the output levels

# Losses

Implement losses from the paper https://arxiv.org/pdf/2006.04388.pdf . Since tensorflow doesn't allow applying one loss on multiple outputs, implement losses as layers and use model output as loss

Loss between predicted and true boxes

In [ ]:
@keras.utils.register_keras_serializable()
class CIouLoss(keras.layers.Layer):
    
    def __init__(self, eps: float = 1e-6, **kwargs):
        super().__init__(**kwargs)
        ...

    def call(self, y_true: keras.KerasTensor, y_pred: keras.KerasTensor) -> keras.KerasTensor:
        ...
        
    def get_config(self) -> dict:
        ...

Loss between predicted and true labels

In [ ]:
@keras.utils.register_keras_serializable()
class FocalLoss(keras.layers.Layer):
    
    def __init__(self, gamma: float = 2, **kwargs):
        super().__init__(**kwargs)
        ...
        
    def call(self, y_true: keras.KerasTensor, y_pred: keras.KerasTensor) -> keras.KerasTensor:
        ...
    
    def get_config(self) -> dict:
        ...

Loss between predicted distance decomposition and actual (see article for details)

In [ ]:
@keras.utils.register_keras_serializable()
class DistributionFocalLoss(keras.layers.Layer):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        ...
        
    def call(self, y_true: keras.KerasTensor, y_pred: keras.KerasTensor) -> keras.KerasTensor:
        ...
    
    def get_config(self) -> dict:
        ...

Combined loss. Class loss is applied for every anchor. The rest of the losses are applied for non-background anchors

In [ ]:
@keras.utils.register_keras_serializable()
class DetectionLoss(keras.layers.Layer):
    
    def __init__(
        self,
        anchors: list[np.ndarray | keras.KerasTensor],
        strides: list[int],
        cls_weight: float,
        box_weight: float,
        dfl_weight: float,
        cls_loss: keras.layers.Layer,
        box_loss: keras.layers.Layer,
        dfl_loss: keras.layers.Layer,
        **kwargs
):
        super().__init__(**kwargs)
        ...
        
    def get_config(self) ->dict:
        ...
          
    def call(self, y_true: list[tuple[keras.KerasTensor, keras.KerasTensor]], y_pred: list[tuple[keras.KerasTensor, keras.KerasTensor]]) -> keras.KerasTensor:
        ...

# Training

# Testing

Don't forget to use NMS during prediction [link](https://www.tensorflow.org/api_docs/python/tf/image/non_max_suppression_with_scores)

In [ ]:
def predict(image: np.ndarray, detection_threshold: float, iou_threshold: float) -> tuple[np.ndarray, np.ndarray]:
    """Predicts boxes for the image.

    Returns:
        predicted boxes and their respective scores"""
    ...

Plot some detections on images not present in the dataset